# Ex. 1: Doua zaruri
Se arunca doua zaruri, iar rezultatele sunt notate cu X si Y.

1. Probabilitatea ca suma sa fie cel putin 10, stiind ca primul zar indica un numar par.
2. Probabilitatea ca suma sa fie cel putin 10, stiind ca cel putin unul dintre zaruri indica un numar par.
3. Probabilitatea ca ambele zaruri sa indice numere pare, stiind ca suma lor este 8.
4. Sunt evenimentele $A = \{X + Y \ge 10\}$ si $B = \{X \text{ este par}\}$ independente?

Simulam 100.000 de aruncari, estimam probabilitatile conditionate si le comparam cu valorile exacte.

## Pasul 1: Valorile exacte
Omega are 36 de perechi (x, y), toate la fel de probabile, deci P(E) = cazuri favorabile / 36.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from fractions import Fraction

omega = [(x, y) for x in range(1, 7) for y in range(1, 7)]
print(f"|Omega| = {len(omega)}")

def P_exact(eveniment):
    favorabile = [w for w in omega if eveniment(*w)]
    return Fraction(len(favorabile), len(omega))

def P_cond_exact(eveniment, conditie):
    return P_exact(lambda x, y: eveniment(x, y) and conditie(x, y)) / P_exact(conditie)

In [ ]:
def A(x, y): return x + y >= 10                 # suma >= 10
def B(x, y): return x % 2 == 0                  # primul zar par
def C(x, y): return x % 2 == 0 or y % 2 == 0    # cel putin un zar par
def D(x, y): return x + y == 8                  # suma = 8
def E(x, y): return x % 2 == 0 and y % 2 == 0   # ambele pare

exact = {
    "P(A|B)": P_cond_exact(A, B),
    "P(A|C)": P_cond_exact(A, C),
    "P(E|D)": P_cond_exact(E, D),
}

for nume, val in exact.items():
    print(f"{nume} = {val} = {float(val):.4f}")

## Pasul 2: Simularea
Aruncam cele doua zaruri de 100.000 de ori.

In [ ]:
np.random.seed(42)
N = 100_000

X = np.random.randint(1, 7, size=N)   # primul zar
Y = np.random.randint(1, 7, size=N)   # al doilea zar

## Pasul 3: Probabilitatile conditionate din simulare
P(A|B) ≈ (de cate ori s-au intamplat A si B) / (de cate ori s-a intamplat B)

Adica pastram doar aruncarile in care s-a intamplat B si numaram in cate dintre ele s-a intamplat si A.

In [ ]:
ev_A = X + Y >= 10
ev_B = X % 2 == 0
ev_C = (X % 2 == 0) | (Y % 2 == 0)
ev_D = X + Y == 8
ev_E = (X % 2 == 0) & (Y % 2 == 0)

def P_cond_sim(eveniment, conditie):
    return (eveniment & conditie).sum() / conditie.sum()

simulat = {
    "P(A|B)": P_cond_sim(ev_A, ev_B),
    "P(A|C)": P_cond_sim(ev_A, ev_C),
    "P(E|D)": P_cond_sim(ev_E, ev_D),
}

for nume in exact:
    print(f"{nume}: simulat = {simulat[nume]:.4f}, exact = {float(exact[nume]):.4f}")

Valorile simulate sunt foarte apropiate de cele exacte, deci calculele sunt corecte.

## Pasul 4: Independenta lui A si B
A si B sunt independente daca P(A ∩ B) = P(A) * P(B).

In [ ]:
p_AB = P_exact(lambda x, y: A(x, y) and B(x, y))
p_A = P_exact(A)
p_B = P_exact(B)

print(f"P(A ∩ B)    = {p_AB}  (simulat {(ev_A & ev_B).mean():.4f})")
print(f"P(A) * P(B) = {p_A * p_B}  (simulat {ev_A.mean() * ev_B.mean():.4f})")

1/9 ≠ 1/12, deci A si B **nu** sunt independente. Daca stim ca primul zar e par, creste sansa ca suma sa fie cel putin 10.

## Pasul 5: Grafic

In [ ]:
nume = list(exact.keys())
poz = np.arange(len(nume))

plt.bar(poz - 0.2, [float(exact[n]) for n in nume], 0.4, label="exact")
plt.bar(poz + 0.2, [simulat[n] for n in nume], 0.4, label="simulat")
plt.xticks(poz, nume)
plt.ylabel("probabilitate")
plt.title("Exact vs simulat")
plt.legend()
plt.show()

## Concluzie
- P(A|B) = 2/9
- P(A|C) = 5/27
- P(E|D) = 3/5
- A si B nu sunt independente (1/9 ≠ 1/12)

Simularea confirma valorile calculate.